# 03 · HMM, validation e rischio / HMM, validation and risk

**Input · IT:** Input scalati, blocchi datati e baseline del notebook 02.

**Output · IT:** HMM selezionato, `results/regime_results.csv`, profili e valutazione; confronto di rischio a 5 sedute se attivato.

**Input · EN:** Scaled inputs, dated blocks and baselines from Notebook 02.

**Output · EN:** Selected HMM, `results/regime_results.csv`, profiles and evaluation; a five-session risk comparison if enabled.

**Prima di iniziare / Before starting:** [HMM e transizioni / HMMs and transitions](https://hmmlearn.readthedocs.io/en/stable/tutorial.html), [inferenza / inference](https://hmmlearn.readthedocs.io/en/stable/api.html), [Brier](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html) e [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html). Filtering: dati fino a t / observations through t; smoothing: anche dati successivi / later observations too.

**Avvio / Start:** Jupyter dalla cartella del progetto; import di `core` dalla radice con `settings.json` / Jupyter from the project folder; import `core` from the root containing `settings.json`.


## A · Training e scelta del modello / Training and model selection

### 1. Limitare la griglia / Limit the grid

**IT:** Preparate HMM con 2, 3 o 4 stati e tre seed per ciascun sottoinsieme di input. Contate tutti i tentativi, compresi errori e retry: massimo 40 fit HMM nell’intera pipeline.

**EN:** Prepare HMMs with 2, 3 or 4 states and three seeds for each input subset. Count every attempt, including errors and retries: at most 40 HMM fits across the complete pipeline.


### 2. Adattare gli HMM / Fit the HMMs

**IT:** Implementate `models.fit_hmm_candidates` sul training di un solo sottoinsieme, con Gaussiane diagonali. Conservate i modelli ed esportate `results/hmm_diagnostics.csv`: likelihood, warning, occupazione, varianze e convergenza distinta dal limite d’iterazioni.

**EN:** Implement `models.fit_hmm_candidates` on training for one input subset, using diagonal Gaussians. Retain the models and export `results/hmm_diagnostics.csv`: likelihood, warnings, occupancy, variances and convergence distinguished from the iteration limit.


### 3. Calcolare il filtro continuo / Calculate the continuous filter

**IT:** Implementate `models.filter_probabilities` usando `startprob_` direttamente per il primo dato e poi posterior @ transmat_, combinando densità Gaussiane in modo stabile. Restituite `probability_0…K−1` e `log_predictive_density` datate; al confine usate `previous_posterior @ transmat_` per il primo dato del blocco successivo e verificate i prefissi.

**EN:** Implement `models.filter_probabilities` using `startprob_` directly for the first observation, then posterior @ transmat_, combining Gaussian densities with stable calculations. Return dated `probability_0…K−1` and `log_predictive_density`; at a boundary, use `previous_posterior @ transmat_` for the next block’s first observation and check prefixes.


### 4. Selezionare il modello / Select the model

**IT:** Implementate `models.select_hmm`: seed dal training, numero di stati dalla validation, su input e scaler identici. Verificate diagnostiche e scelta indipendente dal test; l’ordine degli stati resta quello fissato sul training.

**EN:** Implement `models.select_hmm`: seeds from training, state count from validation, using identical inputs and scaler. Check diagnostics and test-independent selection; state order remains fixed from training.


### 5. Integrare l’inferenza / Integrate inference

**IT:** Portate training e filtro in `core/models.py`, mantenendo lo smoothing separato. Verificate budget, probabilità unitarie, continuità fra blocchi e risultati identici dopo ricaricamento.

**EN:** Move training and filtering into `core/models.py`, keeping smoothing separate. Check the fit budget, unit probabilities, continuity across blocks and identical results after reloading.


## B · Profili e validation / Profiles and validation

### 6. Assemblare i risultati / Assemble the results

**IT:** Salvate date, split, label dei tre modelli, probabilità HMM/GMM, confidence, entropia e log-densità in `results/regime_results.csv`. Controllate allineamento e timing: training retrospettivo, validation di selezione e blocco finale già osservato come holdout illustrativo.

**EN:** Save dates, splits, labels from all three models, HMM/GMM probabilities, confidence, entropy and log density to `results/regime_results.csv`. Check alignment and timing: retrospective training, validation used for selection and the already inspected final block as an illustrative holdout.


### 7. Descrivere gli asset / Describe the assets

**IT:** Implementate `validation.build_regime_profiles` sui rendimenti giornalieri calcolati prima del raggruppamento e salvate conteggi, quote, rendimenti, volatilità e correlazioni in `results/regime_profiles.csv`. Controllate unità e gruppi piccoli: sono statistiche contemporanee, non rendimenti di una strategia.

**EN:** Implement `validation.build_regime_profiles` using daily returns calculated before grouping and save counts, shares, returns, volatility and correlations to `results/regime_profiles.csv`. Check units and small groups: these are same-day statistics, not strategy returns.


### 8. Valutare stabilità e incertezza / Evaluate stability and uncertainty

**IT:** Implementate `validation.evaluate_model` per `results/model_comparison.csv`, includendo occupazione, switching, entropia e accordo ARI fra seed su date identiche. Valutate il blocco finale dopo la selezione, senza confondere probabilità concentrate con accuratezza predittiva.

**EN:** Implement `validation.evaluate_model` for `results/model_comparison.csv`, including occupancy, switching, entropy and seed agreement via ARI on identical dates. Evaluate the final block after selection, without treating concentrated probabilities as predictive accuracy.


### 9. Costruire i grafici / Build the charts

**IT:** Implementate `plots.build_charts` per prezzi ribasati, probabilità, entropia, profili e transizioni. Controllate periodi e unità, distinguendo durata teorica degli stati e durata delle label osservate.

**EN:** Implement `plots.build_charts` for rebased prices, probabilities, entropy, profiles and transitions. Check periods and units, distinguishing theoretical state duration from observed label-run duration.


### 10. Integrare la valutazione / Integrate evaluation

**IT:** Portate profili e metriche in `core/validation.py`, i grafici in `core/plots.py`. Verificate join, conteggi, casi brevi o vuoti e ordine degli stati; assegnate nomi economici soltanto dopo i profili training.

**EN:** Move profiles and metrics into `core/validation.py`, and charts into `core/plots.py`. Check joins, counts, short or empty periods and state order; assign economic names only after reviewing training profiles.


## C · Proiezione e confronto opzionale / Projection and optional comparison

### 11. Proiettare la prossima distribuzione / Project the next distribution

**IT:** Implementate `forecast.predict_next_states(model, probabilities)` per ottenere πₜ @ A dal posterior filtrato, salvando la proiezione datata in `results/regime_results.csv`. Verificate ordine e somma unitaria: è una distribuzione sugli stati della prossima seduta, non una previsione di crisi né una label osservata.

**EN:** Implement `forecast.predict_next_states(model, probabilities)` to obtain πₜ @ A from the filtered posterior and save the dated projection to `results/regime_results.csv`. Check order and unit sums: it is a distribution over next-session states, not a crisis forecast or an observed label.


### 12. Confrontare il rischio a 5 sedute - opzionale / Compare five-session risk - optional

**IT:** Se `risk_forecast.enabled`, implementate `forecast.build_risk_targets` e `forecast.evaluate_risk_forecast`.

- **Target:** volatilità dei prossimi 5 rendimenti SPY; soglia al 75° percentile dei target training validi. Escludete le ultime 5 origini quando il target supera il confine del blocco.
- **Modelli:** prevalenza training come baseline; logistic sulla volatilità corrente a 20 sedute; stessa logistic con K−1 probabilità HMM. Adattate sul training e scegliete C sulla validation.
- **Consegna:** `results/forecast_comparison.csv` con Brier, average precision, conteggi e prevalenza. Ripetete il confronto su origini non sovrapposte, usando il passo 5 prefissato.

**EN:** If `risk_forecast.enabled`, implement `forecast.build_risk_targets` and `forecast.evaluate_risk_forecast`.

- **Target:** volatility of the next 5 SPY returns; threshold at the 75th percentile of valid training targets. Drop the last 5 origins when their target crosses a block boundary.
- **Models:** training prevalence as a baseline; logistic regression on current 20-session volatility; the same regression with K−1 HMM probabilities. Fit on training and choose C on validation.
- **Delivery:** `results/forecast_comparison.csv` with Brier, average precision, counts and prevalence. Repeat the comparison on non-overlapping origins using the prespecified five-session step.
